In [1]:
import pandas as pd
import numpy as np

In [2]:
movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")

In [3]:
movies.sample(5)

,movieId,title,genres
2965,3974,"Little Vampire, The (2000)",Adventure|Children
9473,168608,Mudbound (2017),Drama
9600,175781,Der Herr Karl (1961),Comedy
3601,4945,"Enforcer, The (1976)",Crime
6078,41573,"Family Stone, The (2005)",Comedy|Drama|Romance


In [4]:
ratings.sample(5)

,userId,movieId,rating,timestamp
16554,105,30812,4.0,1446572065
71451,460,2747,3.0,1359176295
98371,606,34405,2.5,1180737464
58143,381,597,4.0,1164877296
45110,298,54259,2.5,1447560231


In [5]:
movies.shape

(9742, 3)

In [6]:
ratings.shape

(100836, 4)

In [7]:
movies.columns

Index(['movieId', 'title', 'genres'], dtype='object')

In [8]:
ratings.columns

Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='object')

In [9]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


In [10]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [11]:
movies.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [12]:
ratings.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [13]:
movies.duplicated().sum()

np.int64(0)

In [14]:
ratings.duplicated().sum()

np.int64(0)

In [15]:
ratings["userId"].nunique()

610

In [16]:
movies["movieId"].nunique()

9742

In [17]:
ratings['rating'].value_counts().sort_index()

rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64

In [18]:
most_rated_movies=(
    ratings["movieId"].value_counts().reset_index()
)
most_rated_movies.columns=["movieId","rating_count"]
most_rated_movies=most_rated_movies.merge(movies,on="movieId")
most_rated_movies.sample(5)

,movieId,rating_count,title,genres
4066,4518,4,The Lair of the White Worm (1988),Comedy|Horror
2451,36525,8,Just Like Heaven (2005),Comedy|Fantasy|Romance
1595,412,15,"Age of Innocence, The (1993)",Drama
6615,67888,1,Man on the Flying Trapeze (1935),Comedy
7899,98230,1,10 Years (2011),Comedy|Drama|Romance


In [19]:
avg_rating=(
    ratings.groupby("movieId")["rating"].mean()
    .reset_index(name="avg_rating")
)
most_rated_movies = most_rated_movies.merge(
    avg_rating,on="movieId"
)
most_rated_movies.sample(5)

,movieId,rating_count,title,genres,avg_rating
4167,5102,4,Rookie of the Year (1993),Comedy|Fantasy,2.500000
8825,148238,1,A Very Murray Christmas (2015),Comedy,2.500000
1977,7324,11,Hidalgo (2004),Adventure|Drama,3.318182
6484,128944,1,"Honey, We Shrunk Ourselves (1997)",Action|Adventure|Children|Comedy|Sci-Fi,3.500000
2799,1975,7,Friday the 13th Part 2 (1981),Horror,2.500000


In [20]:
movies["genres"].head(10)

0    Adventure|Animation|Children|Comedy|Fantasy
1                     Adventure|Children|Fantasy
2                                 Comedy|Romance
3                           Comedy|Drama|Romance
4                                         Comedy
5                          Action|Crime|Thriller
6                                 Comedy|Romance
7                             Adventure|Children
8                                         Action
9                      Action|Adventure|Thriller
Name: genres, dtype: object

In [23]:
all_genres=movies["genres"].str.split("|").explode().unique()
all_genres

array(['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy',
       'Romance', 'Drama', 'Action', 'Crime', 'Thriller', 'Horror',
       'Mystery', 'Sci-Fi', 'War', 'Musical', 'Documentary', 'IMAX',
       'Western', 'Film-Noir', '(no genres listed)'], dtype=object)

In [24]:
n=int(input("Enter number of members: "))
members_genres=[]
for i in range(n):
    genres=input(
        f"Enter genres for member {i+1}, separated by commas: "
    ).split(",")
    genres=[genre.strip() for genre in genres]
    members_genres.append(genres)
print(members_genres)

[['comedy'], ['fantasy'], ['drama']]


In [25]:
group_genres=[]
for genres in members_genres:
    group_genres.extend(genres)
print(group_genres)

['comedy', 'fantasy', 'drama']


In [26]:
from collections import Counter
genres_counts=Counter(group_genres)
print(genres_counts)

Counter({'comedy': 1, 'fantasy': 1, 'drama': 1})


In [28]:
from collections import Counter
genre_counts = Counter()
for genres in members_genres:
    genre_counts.update(set(genres))
print(genre_counts)

Counter({'comedy': 1, 'fantasy': 1, 'drama': 1})


In [29]:
def calculate_genre_score(movie_genres):
    score=0
    for genre in movie_genres:
        score+=genre_counts.get(genre,0)
    return score

In [35]:
most_rated_movies["genre_score"]=(
    most_rated_movies["genres"].apply(calculate_genre_score)
)

In [37]:
most_rated_movies[["title","genres","genre_score","avg_rating"]].head(10)

,title,genres,genre_score,avg_rating
0,Forrest Gump (1994),"[Comedy, Drama, Romance, War]",0,4.164134
1,"Shawshank Redemption, The (1994)","[Crime, Drama]",0,4.429022
2,Pulp Fiction (1994),"[Comedy, Crime, Drama, Thriller]",0,4.197068
3,"Silence of the Lambs, The (1991)","[Crime, Horror, Thriller]",0,4.161290
4,"Matrix, The (1999)","[Action, Sci-Fi, Thriller]",0,4.192446
5,Star Wars: Episode IV - A New Hope (1977),"[Action, Adventure, Sci-Fi]",0,4.231076
6,Jurassic Park (1993),"[Action, Adventure, Sci-Fi, Thriller]",0,3.750000
7,Braveheart (1995),"[Action, Drama, War]",0,4.031646
8,Terminator 2: Judgment Day (1991),"[Action, Sci-Fi]",0,3.970982
9,Schindler's List (1993),"[Drama, War]",0,4.225000


In [38]:
recommended_movies=most_rated_movies.sort_values(
    by=["genre_score","avg_rating"],
    ascending=[False,False]
)
recommended_movies[
    ["title","genres","genre_score","avg_rating"]
].head(10)

,title,genres,genre_score,avg_rating
4981,Lesson Faust (1994),"[Animation, Comedy, Drama, Fantasy]",0,5.0
5115,Jonah Who Will Be 25 in the Year 2000 (Jonas q...,[Comedy],0,5.0
5139,Enter the Void (2009),[Drama],0,5.0
5266,Come and See (Idi i smotri) (1985),"[Drama, War]",0,5.0
5485,Lamerica (1994),"[Adventure, Drama]",0,5.0
5581,Heidi Fleiss: Hollywood Madam (1995),[Documentary],0,5.0
5969,Belle époque (1992),"[Comedy, Romance]",0,5.0
6282,"Son of the Bride (Hijo de la novia, El) (2001)","[Comedy, Drama]",0,5.0
6285,Light Years (Gandahar) (1988),"[Adventure, Animation, Fantasy, Sci-Fi]",0,5.0
6287,Moscow Does Not Believe in Tears (Moskva sleza...,"[Drama, Romance]",0,5.0
